In [3]:
# =============================================================================
# 🐾 PAWTY AI BACKEND - GOOGLE COLAB PRO EDITION
# =============================================================================

import os
import sys
import subprocess
import threading
import base64
from io import BytesIO
from PIL import Image
import torch
import asyncio

# --- 1. Installs dependencies at top speed (only during first run) ---
print("🚀 Initializing environment...")
try:
    import diffusers
    import fastapi
    import pyngrok
    print("✅ 依赖库已安装")
except ImportError:
    print("⬇️ Dependency libraries have been installed (diffusers, fastapi, uvicorn, pyngrok)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "diffusers", "transformers", "accelerate", "safetensors",
                           "fastapi", "uvicorn", "pyngrok", "python-multipart", "nest_asyncio"])
    print("✅ Installation complete")

import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from diffusers import StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler

# --- 2. Load AI Model Engine (Pro GPU Edition) ---
class PetStyleService:
    def __init__(self, model_id="Lykon/dreamshaper-8"):
        print(f"🔄 Loading model: {model_id}...")

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🖥️  Operating Equipment: {self.device} (should be cuda)")

        if self.device == "cpu":
            print("⚠️Warning: No GPU detected! Rendering speed will be extremely slow. Please check your runtime settings.")

        # Using float16 precision (fastest on Colab Pro A100/T4)
        self.pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            use_safetensors=True
        )

        # Use the DPM++ scheduler (for better quality and fewer steps)
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++"
        )

        self.pipe = self.pipe.to(self.device)
        self.pipe.enable_attention_slicing() #Graphics Memory Optimization
        print("✨ Model loaded and ready to go!")

    def generate(self, init_image, style_prompt, species, strength=0.75):
        # Preprocess images: Resize to 512x512 to ensure speed and VRAM efficiency.
        init_image = init_image.convert("RGB").resize((512, 512))

        # Your “Strict Style” Prompt Logic
        full_prompt = (
            f"({style_prompt}:1.4), masterpiece, best quality, 8k, "
            f"(cute {species}), (animal only), detailed fur, cinematic lighting"
        )
        negative_prompt = "human, person, man, woman, hands, feet, text, watermark, bad anatomy, blur, lowres"

        with torch.autocast("cuda"):
            result = self.pipe(
                prompt=full_prompt,
                image=init_image,
                strength=strength,      # # 0.75 indicates a 75% redraw
                guidance_scale=9.0,     # Force the model to follow the prompt
                negative_prompt=negative_prompt,
                num_inference_steps=25  # DPM++ 25 steps are enough
            ).images[0]

        return result

# --- 3. Configure FastAPI Service ---
app = FastAPI()

# Allow cross-origin requests (resolves local webpage access issues)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Style Mapping Table (Corresponding to IDs in HTML)
STYLE_MAP = {
    "1": {"p": "Oil Painting, thick strokes, textured canvas", "s": 0.75},
    "2": {"p": "Pixar style, disney 3d render, cute, vibrant", "s": 0.75},
    "3": {"p": "Cyberpunk, neon lights, mechanical parts, sci-fi", "s": 0.80},
    "4": {"p": "Pencil sketch, graphite, monochrome, rough lines", "s": 0.65},
    "5": {"p": "Ghibli style, anime, vibrant colors, detailed background", "s": 0.75}
}

service = PetStyleService() # Initialize the model

@app.post("/stylize")
async def process_image(
    style_id: str = Form(...),
    species: str = Form(...),
    image: UploadFile = File(...)
):
    print(f"📩 request received: {species} | style: {style_id}")

    # Read uploaded image
    img_bytes = await image.read()
    input_pil = Image.open(BytesIO(img_bytes))

    # Retrieve Style Configuration
    style = STYLE_MAP.get(style_id, STYLE_MAP["1"])

    # Image generation
    output_pil = service.generate(input_pil, style["p"], species, style["s"])

    # To Base64
    buffered = BytesIO()
    output_pil.save(buffered, format="PNG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

    return {"status": "success", "image_base64": img_b64}

# --- 4. Launch Server (Fixed Version for L4/Colab) ---
import uvicorn
import nest_asyncio

# ================= Enter your TOKEN here =================
NGROK_TOKEN = "36lRCtcHBQ8soISUzEnvrC9uYMz_4MHNArj68JaNrP8zDNdaU"
# ======================================================

# 1. Set Ngrok
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill() # 关闭旧隧道

# 2. Create Tunnel (Port 8011)
tunnel = ngrok.connect(8011)
public_url = tunnel.public_url

print(f"\n✅ =============================================")
print(f"🎉 Server is up and running (L4 GPU acceleration enabled)!")
print(f"👉 Please copy this URL to index.html.: {public_url}")
print(f"=============================================\n")

# 3.  Key Fix for Resolving Colab Errors:
nest_asyncio.apply()

# Use the Config and Server objects with await to start
config = uvicorn.Config(app, port=8011, host="0.0.0.0")
server = uvicorn.Server(config)
await server.serve()  

/usr/local/lib/python3.12/dist-packages/IPython/core/compilerop.py:101: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)


🚀 正在初始化环境...
✅ 依赖库已安装
🔄 正在加载模型: Lykon/dreamshaper-8...
🖥️  运行设备: cuda (应该是 cuda)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.


✨ 模型加载完毕，准备就绪！

✅ =============================================
🎉 服务器已启动 (L4 GPU 加速中)！
👉 请复制这个 URL 到 index.html: https://dwight-unexpectant-aloofly.ngrok-free.dev



...

  0%|          | 0/20 [00:00<?, ?it/s]

INFO:     207.38.249.232:0 - "POST /stylize HTTP/1.1" 200 OK
📩 收到请求: cat | 风格: 5


  0%|          | 0/18 [00:00<?, ?it/s]

INFO:     207.38.249.232:0 - "POST /stylize HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1182]
